# Phase 2 — Hybrid Ensemble Model: Hypertension Prediction (RIPAS Dataset)
**Triage IQ — Predicting Hypertension from Emergency Department Triage Data**

This notebook implements a soft-voting ensemble combining XGBoost and Gradient Boosting.  
The purpose is to test whether combining the two best Phase 2 models improves over either individually.

**Dataset:** `ripas_dataset.csv`  
**Target variable:** `HTN/CHD` (clinically recorded, binary)  
**Ensemble:** `VotingClassifier(XGBoost + GradientBoosting, voting='soft', weights=[5,1])`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RandomizedSearchCV,
    RepeatedStratifiedKFold, cross_validate
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("ripas_dataset.csv")

print("Shape:", df.shape)
df.head()

## 3. Data Cleaning

In [ ]:
# Strip trailing 'Y' from age strings (e.g. '45Y' -> 45)
df['age'] = pd.to_numeric(
    df['age'].astype(str).str.replace('Y', '', regex=False),
    errors='coerce'
)

# Convert vital-sign and stay columns to numeric
numeric_cols = [
    'Blood pressure diastole',
    'blood pressure/systolic',
    'pulse rate',
    'respiratory rate',
    'SPO2',
    'Temperature',
    'PAIN SCORE',
    'STAYS IN DAYS'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Encode binary comorbidity / outcome flags (NO -> 0, named value -> 1)
binary_cols = ['HTN/CHD', 'DM', 'CKD', 'ADMISSION', 'ICU', 'NIV/VENT', 'INOTROPE', 'DEATH']
for col in binary_cols:
    df[col] = df[col].replace({'NO': 0, 'HTN/CHD': 1, 'DM': 1, 'CKD': 1})
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Encode gender
df['gender'] = df['gender'].map({'M': 1, 'F': 0})

print("Cleaned dataset shape:", df.shape)
df.head()

## 4. Define Target Variable

In [ ]:
# Drop rows where the target label is missing
df = df[df['HTN/CHD'].notna()].reset_index(drop=True)

print("Remaining NaN in target:", df['HTN/CHD'].isna().sum())

X = df.drop('HTN/CHD', axis=1)
y = df['HTN/CHD']

print("\nClass distribution:")
print(y.value_counts())
print("\nClass proportions:")
print(y.value_counts(normalize=True).round(3))

## 5. Train/Test Split

In [ ]:
# 80/20 stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print("\nTraining class distribution:")
print(y_train.value_counts())

## 6. Class Distribution and SMOTE Effect

In [ ]:
# Visualise class imbalance before any resampling
fig, ax = plt.subplots(figsize=(5, 3))
y_train.value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xticklabels(['No HTN', 'HTN'], rotation=0)
ax.set_title("Training Set Class Distribution (Before SMOTE)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = ['site of pain']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler())
        ]), numerical_cols),

        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

In [ ]:
# Illustrate SMOTE effect by fitting preprocessor + SMOTE on training data
X_pre = preprocessor.fit_transform(X_train)
_, y_res = SMOTE(sampling_strategy=0.8, random_state=42).fit_resample(X_pre, y_train)

print("Class distribution after SMOTE:")
print(pd.Series(y_res).value_counts())

fig, ax = plt.subplots(figsize=(5, 3))
pd.Series(y_res).value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xticklabels(['No HTN', 'HTN'], rotation=0)
ax.set_title("Training Set Class Distribution (After SMOTE, sampling_strategy=0.8)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 7. Hybrid Ensemble Pipeline

Initial weights `[5, 1]` favour XGBoost strongly — XGBoost was the stronger individual model  
from Phase 1 benchmarking. The weight ratio is tuned via `RandomizedSearchCV`.

In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)
gb_model = GradientBoostingClassifier(random_state=42)

ensemble = VotingClassifier(
    estimators=[('xgb', xgb_model), ('gb', gb_model)],
    voting='soft',
    weights=[5, 1]
)

pipeline_ensemble = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(sampling_strategy=0.8, random_state=42)),
    ('model',        ensemble)
])

## 8. Baseline Evaluation

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scoring = {'recall': 'recall', 'roc_auc': 'roc_auc', 'precision': 'precision', 'f1': 'f1'}

cv_results = cross_validate(pipeline_ensemble, X_train, y_train, cv=rskf, scoring=scoring)

print("Repeated Stratified CV — Baseline Hybrid Ensemble:")


In [ ]:
for key in cv_results:
    if key.startswith("test_"):
        mean_score = np.mean(cv_results[key])
        std_score  = np.std(cv_results[key])
        print(f"{key[5:]:12s}: {mean_score:.4f} ± {std_score:.4f}")

In [ ]:
pipeline_ensemble.fit(X_train, y_train)
y_prob_base = pipeline_ensemble.predict_proba(X_test)[:, 1]

y_pred_base_05 = pipeline_ensemble.predict(X_test)
y_pred_base    = (y_prob_base >= 0.35).astype(int)

for label, y_pred in [("Threshold = 0.5", y_pred_base_05), ("Threshold = 0.35", y_pred_base)]:
    print(f"--- BASELINE {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_base):.3f}")
    print(classification_report(y_test, y_pred))

## 9. Hyperparameter Tuning — RandomizedSearchCV

Both sub-model parameter spaces and the ensemble weights are searched jointly.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

param_dist = {
    'model__xgb__n_estimators':  [100, 200, 300],
    'model__xgb__max_depth':     [3, 5, 7],
    'model__xgb__learning_rate': [0.01, 0.1, 0.2],
    'model__xgb__subsample':     [0.8, 1.0],
    'model__gb__n_estimators':   [100, 200],
    'model__gb__max_depth':      [3, 5],
    'model__gb__learning_rate':  [0.05, 0.1],
    'model__gb__subsample':      [0.8, 1.0],
    'model__weights':            [(1, 1), (2, 1), (3, 1), (5, 1)]
}

search = RandomizedSearchCV(
    pipeline_ensemble,
    param_distributions=param_dist,
    n_iter=25,
    cv=cv,
    scoring='recall',
    n_jobs=-1,
    random_state=42,
    verbose=0
)
search.fit(X_train, y_train)

print("Best recall (RandomCV):", round(search.best_score_, 3))
print("Best parameters:",        search.best_params_)

## 10. Test Set Evaluation — Tuned Ensemble

In [ ]:
best_model = search.best_estimator_
y_prob     = best_model.predict_proba(X_test)[:, 1]

y_pred_default = best_model.predict(X_test)
y_pred         = (y_prob >= 0.35).astype(int)

for label, y_p in [("Threshold = 0.5", y_pred_default), ("Threshold = 0.35", y_pred)]:
    print(f"--- TUNED ENSEMBLE {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_p):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")
    print(classification_report(y_test, y_p))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_p, title in zip(
    axes,
    [y_pred_default, y_pred],
    ["Hybrid Ensemble (Threshold 0.5)", "Hybrid Ensemble (Threshold 0.35)"]
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_p),
        display_labels=["No HTN", "HTN"]
    ).plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Hybrid Ensemble")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Threshold Analysis

The precision-recall trade-off is evaluated across a range of thresholds to justify the 0.35 selection.  
In a screening context, the cost of a missed hypertensive patient (false negative) outweighs the cost of a false positive.

In [ ]:
thresholds = [0.5, 0.45, 0.40, 0.35, 0.30, 0.25]

threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    threshold_results.append({
        "Threshold": t,
        "Recall":    round(report.get("1.0", report.get(1, {})).get("recall",    0), 3),
        "Precision": round(report.get("1.0", report.get(1, {})).get("precision", 0), 3),
        "F1":        round(report.get("1.0", report.get(1, {})).get("f1-score",  0), 3),
        "Accuracy":  round(accuracy_score(y_test, y_pred_t), 3)
    })

threshold_df = pd.DataFrame(threshold_results)
print("Threshold sweep results:")
display(threshold_df)

In [ ]:
# Visualise recall vs precision across thresholds
plt.figure(figsize=(7, 4))
plt.plot(threshold_df["Threshold"], threshold_df["Recall"],    marker='o', label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker='s', label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["F1"],        marker='^', label="F1")
plt.axvline(0.35, color='red', linestyle='--', alpha=0.7, label="Selected threshold (0.35)")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Recall / Precision / F1 vs Classification Threshold")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
os.makedirs("results", exist_ok=True)

summary = pd.DataFrame([
    {"Stage": "Baseline (0.35)",  "Accuracy": accuracy_score(y_test, y_pred_base),
     "ROC-AUC": roc_auc_score(y_test, y_prob_base),
     "Recall":  recall_score(y_test, y_pred_base)},
    {"Stage": "Tuned (0.35)",     "Accuracy": accuracy_score(y_test, y_pred),
     "ROC-AUC": roc_auc_score(y_test, y_prob),
     "Recall":  recall_score(y_test, y_pred)},
])

summary.to_excel("results/hybrid_ripas_summary.xlsx", index=False)
print("Saved: results/hybrid_ripas_summary.xlsx")
display(summary)